# Tutorial: Electron Flow Diagram (EFD)

This Jupyter notebook guides the user on creating an electron flow diagram (EFD) using example data in elektrapy.

In [ ]:
# Setting up elektrapy usage

import sys
sys.path.append("../src/")

from elektrapy.preprocessing import preprocess_data, get_network_df
from elektrapy.efd import (
    get_node_df,
    get_efd,
    get_rti_plot
)

In [ ]:
import os

import pandas as pd

import plotly.express as px

## 1. Setup

Define data folders paths and global variables:

In [ ]:
# Global grouping variable
COLOR_VAR = "type"

# Data directory
DATA_DIR = "../data/example/"

## 2. Preprocessing

### 2.1. Metadata

In [ ]:
metadata_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "metadata.csv"
    )
)
metadata_df

### 2.2. Functional annotation data

**IMPORTANT**: elektrapy uses the output from [bigecyhmm](https://github.com/ArnaudBelcour/bigecyhmm) (Belcour et al., 2025).

The common pipeline consists in the functional annotation of metagenome-assembled genomes (MAGs) using bigecyhmm.
With the predicted redox functions, elektrapy then builds an electron flow diagram (EFD) for visualizing how the electrons are transferred in the given samples, thus facilitating the identification of the major donors and acceptors.

In [ ]:
pathway_df = pd.read_table(
    os.path.join(
        DATA_DIR,
        "bigecyhmm",
        "pathway_presence.tsv"
    )
)
pathway_df

### 2.3. Network dataframe

The basic data architecture that elektrapy uses is the network dataframe, in which the links between electron donors and acceptors are defined.
This dataframe can be obtained by grouping at different levels, for example by genome or sample.

In [ ]:
results_df_genome = preprocess_data(
    df=pathway_df,
    group_var="genome_id"
)
network_df_genome = get_network_df(
    results_df=results_df_genome,
    group_var="genome_id"
)
network_df_genome

In [ ]:
results_df_sample = preprocess_data(
    df=pathway_df,
    group_var="sample_id",
    mapping=dict(zip(metadata_df["genome_id"], metadata_df["sample_id"]))
)
network_df_sample = get_network_df(
    results_df=results_df_sample,
    group_var="sample_id"
)
network_df_sample

Finally, it is recommended to use a variable defined in the input metadata to group the results, which will be used later on for coloring the EFD.

In [ ]:
network_df_group = pd.merge(
    left=network_df_sample,
    right=metadata_df[["sample_id", COLOR_VAR]].drop_duplicates(),
    how="inner",
    on="sample_id"
)

network_df_group = network_df_group\
    .groupby([COLOR_VAR, "source", "target"], as_index=False)\
    ["value"].sum()

network_df_group

## 3. Electron Flow Diagram (EFD)

### 3.1. Redox potentials

Standard redox potentials are required to sort the nodes on the vertical axis.
These values can be defined by the user, but elektrapy provides several values at pH 7 to use.

In [ ]:
redox_df = pd.read_csv("../data/redox-potentials.csv")

# Skip first row containing the units
redox_df = redox_df.iloc[1:]

redox_df

### 3.2. Node data

The ordering of the nodes in the resulting EFD is achieved by the following:

* On the horizontal axis, a redox tendency index (RTI) is calculated, basically representing the proportion of input records in which the node participates as an electron donor or acceptor, ranging from -1 (pure donor) to +1 (pure acceptor).

* On the vertical axis, the mean of the corresponding standard redox potentials at pH7 is used, thus aggregating the redox potential for nodes that can participate in multiple reactions.

In [ ]:
# Get node information (RTI and redox potentials)
# NOTE: it is recommended to aggregate by sample, so that interactions between
# species within the same sample can be captured. It is not recommended to use
# the grouped data since that may group dissimilar samples together
node_df = get_node_df(
    network_df=network_df_sample,
    redox_df=redox_df,
    group_var="sample_id"
)
node_df

In [ ]:
fig = px.scatter(
    data_frame=node_df,
    x="redox_index",
    y="transformed_potential",
    color="redox_index",
    hover_name="node",
    template="plotly_white"
)

# Invert potentials to go from most negative to most positive
fig["layout"]["yaxis"]["autorange"] = "reversed"

fig.show()

### 3.3. EFD

In [ ]:
# Manually define the colors for each category in COLOR_VAR
link_color_map = {
    "surface": "#FFC349",
    "aquifer": "#97DDE9",
    "mine": "#525EA7",
    "reservoir": "#EB7F31"
}

fig = get_efd(
    network_df=network_df_group,
    node_df=node_df,
    color_var=COLOR_VAR,
    link_color_map=link_color_map,
    highlight_node="H2",
    link_alpha=0.1
)
fig.show()

### 3.4. Redox index with grouping variable

In [ ]:
# Add sample ID for getting the redox tendency index
network_df_group = pd.merge(
    left=network_df_sample,
    right=metadata_df[["sample_id", COLOR_VAR]].drop_duplicates(),
    how="inner",
    on="sample_id"
)

fig = get_rti_plot(
    network_df=network_df_group,
    group_var=COLOR_VAR,
    link_color_map=link_color_map
)
fig.show()